# Criteo attribution reliability walkthrough

This notebook exercises `mta-audit` on the public **Criteo Attribution Modeling** dataset
(Diemert et al., 2017). The file is licensed CC-BY-NC-SA-4.0 and is **not** bundled with
the package.

> Attribution is observational and does not establish incrementality or causal impact.

Obtain the dataset from Criteo / Hugging Face, then either place it at
`./data/criteo_attribution_dataset.tsv.gz` or set `download=True` below.

## 1. About the dataset

- ~16.5 million **impressions** over 30 days, ~45k conversions, ~700 anonymized campaigns.
- Each row is a display impression. The `conversion` column means *this impression was followed by a conversion within 30 days*, **not** "this row is a conversion event".
- `mta-audit` therefore uses `CriteoAttributionAdapter`: campaign → channel, and one conversion event per `(uid, conversion_id)` on the last impression at or before `conversion_timestamp`.
- There are **no named media channels** (search, social, etc.). Credit is campaign-level.
- Timestamps are seconds from the first impression (`timestamp` starts at 0).

## 2. Load Criteo

In [ ]:
from mta_audit import MTAAudit, build_journeys
from mta_audit.attribution import (
    FirstTouchAttribution,
    LastTouchAttribution,
    LinearAttribution,
    MarkovAttribution,
)
from mta_audit.datasets import CriteoAttributionAdapter, DatasetNotAvailableError, load_criteo_attribution
from mta_audit.datasets.evaluation import attribution_share_drift, rank_correlation
from mta_audit.simulation import corrupt

# Explicit representative sample. Pass sample_users=None after a local full-file download
# if you want every user; that is much heavier. This is not silent downsampling.
SAMPLE_USERS = 20000
try:
    events = load_criteo_attribution(sample_users=SAMPLE_USERS, random_state=42, download=False)
except DatasetNotAvailableError as exc:
    print(exc)
    events = CriteoAttributionAdapter("./data/criteo_attribution_dataset.tsv.gz").load()
events.head()

## 3. Explore advertising events

In [ ]:
print(events.dtypes)
print("rows", len(events), "users", events["user_id"].nunique(), "campaigns", events["channel"].nunique())
print("conversion events (adapter-constructed)", int(events["conversion"].sum()))
events["click"].value_counts(dropna=False).head()

## 4. Build journeys

In [ ]:
journeys = build_journeys(events, lookback_window="30D")
journeys.summary

## 5. Examine converter / non-converter paths

These statistics are observational. Longer converter paths can reflect selection,
tracking coverage, or browsing intensity — not causal lift.

In [ ]:
s = journeys.summary
print(f"Converters: {s.converter_count:,}  Non-converters: {s.non_converter_count:,}")
print(f"Average path length converters={s.converter_average_path_length:.2f} non-converters={s.non_converter_average_path_length:.2f}")
print(f"Unique paths={s.unique_path_count:,} repeated-touch rate={s.repeated_touch_rate:.1%}")

## 6–9. First-touch, last-touch, linear, and Markov attribution

In [ ]:
first = FirstTouchAttribution().fit(journeys).attribute()
last = LastTouchAttribution().fit(journeys).attribute()
linear = LinearAttribution().fit(journeys).attribute()
markov = MarkovAttribution().fit(journeys).attribute()
for result in (first, last, linear, markov):
    print(result.model, "share sum", round(result.data["share"].sum(), 6))
    display(result.data.head())

## 10. Compare models

If campaign rankings move a lot across methods, budget recommendations based on a
single MTA rule should be treated cautiously. That is **instability**, not proof that
one model is "wrong".

In [ ]:
audit = MTAAudit(data=events, lookback_window="30D")
disagreement = audit.run(
    attribution_models=["first_touch", "last_touch", "linear", "markov"],
    checks=["model_disagreement"],
)
print(disagreement.summary())
disagreement.to_dataframe()

## 11. Audit conversion-window sensitivity

In [ ]:
window = audit.check_conversion_window(windows=[1, 3, 7, 14, 30], model="linear")
print(window.findings[0].message)
window.findings[0].details["metrics"].channel_comparison.head()

## 12. Audit path sparsity

In [ ]:
sparsity = audit.run(checks=["path_sparsity"], attribution_models=["linear"])
print(sparsity.summary())

## 13. Audit channel concentration

In [ ]:
concentration = audit.run(checks=["channel_concentration"], attribution_models=["linear"])
print(concentration.summary())

## 14. Delayed-converter contamination

Criteo already records `conversion_timestamp`. The adapter stores that time on the
constructed conversion event. A 7-day vs 30-day comparison therefore classifies users
by time from first impression to conversion, matching the published 30-day outcome
horizon rather than inventing a separate conversion process.

In [ ]:
contamination = audit.check_delayed_converter_contamination(short_window=7, reference_window=30)
print(contamination.findings[0].message)
contamination.findings[0].details["metrics"]

## 15–16. Simulate 10% identity loss and 20% touchpoint loss

In [ ]:
identity = corrupt(events, identity_loss=0.10, random_state=42)
touch = corrupt(events, touchpoint_loss=0.20, random_state=42)
print(dict(identity.metadata.row_counts))
print(dict(touch.metadata.row_counts))

## 17. Compare clean vs corrupted results

If identity resolution is incomplete, credit often shifts toward campaigns that appear
closer to conversion on the remaining fragments. Treat that as a **tracking-sensitivity**
finding, not as evidence that those campaigns caused incremental conversions.

In [ ]:
clean_report = MTAAudit(data=events).run(
    attribution_models=["first_touch", "last_touch", "linear", "markov"],
    checks=["model_disagreement", "path_sparsity", "data_quality"],
)
id_report = MTAAudit(data=identity.to_dataframe()).run(
    attribution_models=["linear", "markov"],
    checks=["model_disagreement", "path_sparsity", "data_quality"],
)
print("clean reliability", clean_report.score.value, "identity-loss reliability", id_report.score.value)
print("linear share TV distance", attribution_share_drift(clean_report.attribution["linear"], id_report.attribution["linear"]))
print("linear rank corr", rank_correlation(clean_report.attribution["linear"], id_report.attribution["linear"]))

## 18. Generate a final reliability report

In [ ]:
report = audit.run(
    attribution_models=["first_touch", "last_touch", "linear", "markov"],
    conversion_windows=[1, 7, 14, 30],
    checks=[
        "data_quality",
        "conversion_window",
        "converter_contamination",
        "channel_concentration",
        "path_sparsity",
        "model_disagreement",
    ],
    simulations={"identity_loss": {"rates": [0.10], "models": ["linear"]}},
)
print(report.summary())
report.to_dataframe()

## 19. Business implications

- Use the reliability score as a **gate** before putting campaign credit into budget or bidding conversations.
- When first-touch and last-touch ranks diverge, a single-number MTA dashboard can change the apparent winner without any change in delivery.
- Window sensitivity on a 30-day display dataset often flags campaigns whose credit depends on long post-impression delays.
- Identity-loss simulations ask: if the identity graph were 10–20% worse, would the same campaigns still look like they "won"? If not, do not treat current ranks as operationally robust.
- None of these results replace incrementality tests (geo experiments, PSA, conversion lift).

## 20. Methodological limitations

- Adapter conversions are reconstructed from `conversion_id` / `conversion_timestamp`. Alternative reconstructions (click-only paths, Criteo `attribution` flag) would change credit.
- Campaign IDs are not human-readable channels.
- Default runs use an **explicit** user sample for interactive analysis; full-file Markov on 16.5M rows is a batch job (`sample_users=None`).
- Delayed-converter rates inherit Criteo's 30-day observation cap; conversions after day 30 are invisible.
- Simulated missingness is not Criteo's true identity or pixel-loss process.
- Attribution ≠ incrementality.